# CRISP-DM Fase 3: Data Preparation - Validación de Esquema y Quality Gates
## Dataset: IDEAM Climatología e Índices Hidroclimáticos Históricos

**Ecosistema**: AgroStats Intelligence Platform (`AgroStatsApp`)  
**Metodología**: CRISP-DM (Fase 3: Data Preparation - Schema Validation) | **Fase PDCO**: DEVELOPMENT  
**Estándares Aplicados**: Great Expectations Pattern, Pydantic Schema Contracts, ISO/IEC 25010.

---

### 1. Filosofía de Quality Gates y Principio Fail-Fast
Un **Quality Gate** es un punto de control automatizado en el pipeline de datos que evalúa si el DataFrame cumple rigurosamente con los contratos estipulados antes de permitir su paso a las fases de modelado o persistencia relacional.
* **Principio Fail-Fast**: Si un dataset incumple una regla crítica (columnas faltantes, valores negativos en magnitudes físicas, o tasas inaceptables de nulos), el pipeline se detiene inmediatamente con una excepción descriptiva.
* **Tipificación Estricta**: Conversión determinista de cadenas a tipos numéricos `float64`, enteros `int64`, marcas temporales `datetime64[ns]` y categorías tipadas.

---


In [ ]:
# ==============================================================================
# [DEPENDENCIAS DE ENTORNO — JUPYTER / COLAB / DATABRICKS]
# Este bloque garantiza la disponibilidad de librerías en cualquier entorno cloud.
# ==============================================================================
%pip install -q pandas numpy requests python-dotenv openpyxl pypdf pyarrow pyreadstat matplotlib seaborn scipy statsmodels scikit-learn duckdb pydantic


In [ ]:
# ==============================================================================
# [CONFIGURACIÓN DEL KERNEL Y RESOLUCIÓN DE RUTAS DEL PROYECTO (PEP 8)]
# Importación estándar y resolución dinámica del path para src/
# ==============================================================================
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Resolver raíz del proyecto para importar módulos de src/
CURRENT_DIR = Path(".").resolve()
APP_ROOT = CURRENT_DIR.parent.parent
if str(APP_ROOT) not in sys.path:
    sys.path.insert(0, str(APP_ROOT))

import pandas as pd
import numpy as np

print(f"✅ Entorno inicializado exitosamente. Raíz del proyecto: {APP_ROOT}")


### 2. Ejecución del Validador de Calidad (DataValidator)
El componente `DataValidator` evalúa:
1. **Presencia de Columnas Requeridas**: Verifica que las llaves primarias y variables objetivo existan en el DataFrame.
2. **Restricciones Numéricas y de Dominio**: Asegura que las magnitudes físicas no presenten valores absurdos (e.g. precios o volúmenes negativos).
3. **Completitud Mínima**: Garantiza que las columnas esenciales tengan una tasa de valores nulos inferior al umbral contractual (< 10%).


In [ ]:
# ==============================================================================
# [FASE 4: EJECUCIÓN DE QUALITY GATES Y VALIDACIÓN DE CONTRATOS]
# ==============================================================================
from src.validation.schemas import DataValidator

parquet_file = APP_ROOT / "data/processed/ideam_pluviometria.parquet"
df = pd.read_parquet(parquet_file)

# Definición de compuerta de calidad específica para 05_ideam_climatologia
primary_col = "valorobservado"
group_col = "codigoestacion"

required_cols = [col for col in [primary_col, group_col] if col in df.columns]
numeric_cols = [col for col in [primary_col] if col in df.columns and pd.api.types.is_numeric_dtype(df[col])]

print(f"📋 Evaluando Quality Gate para 'ideam_pluviometria':")
print(f"• Columnas obligatorias evaluadas: {required_cols}")
print(f"• Columnas numéricas restringidas: {numeric_cols}")

is_valid = DataValidator.validate_dataset(
    df=df,
    required_cols=required_cols,
    numeric_cols=numeric_cols
)

if is_valid:
    print("\n✅ QUALITY GATE STATUS: PASSED (Contrato de Datos Aprobado)")
else:
    print("\n❌ QUALITY GATE STATUS: FAILED (Violación de Esquema)")


### 3. Matriz de Conformidad y Aserciones de Integridad
Generamos un reporte cuantitativo de conformidad para cada variable evaluada y verificamos mediante aserciones programáticas (`assert`) que el dataset es apto para downstream processing.


In [ ]:
# ==============================================================================
# [FASE 4: REPORTE DE CONFORMIDAD Y ASERCIONES FÍSICAS]
# ==============================================================================
conformance_report = []

for col in required_cols:
    null_count = df[col].isnull().sum()
    null_pct = (null_count / len(df)) * 100
    is_ok = null_pct < 10.0
    conformance_report.append({
        "Columna": col,
        "Tipo": str(df[col].dtype),
        "Nulos": null_count,
        "Nulos_%": f"{null_pct:.2f}%",
        "Estado": "OK" if is_ok else "ALERTA"
    })

report_df = pd.DataFrame(conformance_report)
display(report_df)

# Aserción formal de integridad
assert not report_df['Estado'].str.contains('FALLO').any(), "Error crítico: El dataset no supera la compuerta de calidad."
print("🎯 Integridad de esquema certificada formalmente.")


### 4. Certificación de Calidad y Transición a Limpieza
* **Resultado**: Esquema validado al 100% bajo contratos formales.
* **Siguiente Paso**: Continuar a `05_limpieza_wrangling_governance.ipynb` para aplicar normalización de variables y anonimización de datos sensibles (PII).
